In [2]:
# ============================================================
# Imports + Config
# ============================================================
import json, re, os
from collections import defaultdict
from tqdm import tqdm
import spacy
from openai import OpenAI


base_dir = os.getcwd()
DATA_PATH   = os.path.join(base_dir, "CUADv1.json")
GPT_TRIPLES = os.path.join(base_dir, "gpt_triples_checkpoint.json")
NUM_CONTRACTS  = 50
LEGAL_MODALS   = {"shall","must","will","may","should","cannot","shall not","must not"}
PLACEHOLDER_RE = re.compile(r'\[[\s●\*]+\]', re.MULTILINE)


nlp    = spacy.load("en_core_web_trf")

with open(DATA_PATH) as f:
    data = json.load(f)["data"]

with open(GPT_TRIPLES) as f:
    gpt_raw = json.load(f)
gpt_all = [t for triples in gpt_raw.values() for t in triples]

print(f"CUAD: {len(data)} contracts")
print(f"GPT triples: {len(gpt_all)}")

CUAD: 510 contracts
GPT triples: 995


In [3]:
# ============================================================
# Helpers
# ============================================================
def clean_text(text):
    text = PLACEHOLDER_RE.sub(" ", text)
    return re.sub(r'\s+', ' ', text).strip()

def is_valid(text):
    if not text or len(text.strip()) < 10: return False
    if re.fullmatch(r'\[.*?\]', text.strip()): return False
    return True

def normalize(text):
    text = text.lower().strip()
    text = re.sub(r'\b(the|a|an|its|their|our|this|such)\b', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def get_lemma(pred):
    stop = {"shall","must","will","may","should","cannot","not","be","been",
            "have","had","is","are","was","were","to","directly","indirectly",
            "hereby","promptly","immediately","solely","do"}
    tokens = [t for t in pred.lower().split() if t not in stop]
    return tokens[-1] if tokens else pred.lower()

def get_span(tok, doc):
    start, end = tok.i, tok.i + 1
    for c in tok.subtree:
        if c.dep_ in ("det","amod","compound","poss"):
            start = min(start, c.i)
            end   = max(end, c.i + 1)
    return doc[start:end].text

def extract_openie(text, clause_type, contract):
    doc, triples = nlp(text), []
    for sent in doc.sents:
        for token in sent:
            if token.pos_ not in ("VERB","AUX"): continue
            subj_tok = next((c for c in token.children if c.dep_ in ("nsubj","nsubjpass")), None)
            if not subj_tok: continue
            obj_tok = next((c for c in token.children if c.dep_ == "dobj"), None)
            if not obj_tok:
                for c in token.children:
                    if c.dep_ == "prep":
                        obj_tok = next((g for g in c.children if g.dep_ == "pobj"), None)
                        if obj_tok: break
            if not obj_tok: continue
            pred_parts = [c.text for c in token.lefts if c.dep_ in ("aux","auxpass","neg")]
            pred_parts.append(token.text)
            predicate = " ".join(pred_parts)
            if not any(m in predicate.lower() for m in LEGAL_MODALS): continue
            subj_text = get_span(subj_tok, doc)
            obj_text  = get_span(obj_tok, doc)
            if len(subj_text.strip()) < 2 or len(obj_text.strip()) < 2: continue
            triples.append({
                "subject": subj_text, "predicate": predicate,
                "object": obj_text, "contract": contract,
                "clause_type": clause_type, "raw_text": text, "source": "openie"
            })
    return triples

def triples_match(t1, t2):
    s1, s2 = normalize(t1["subject"]),  normalize(t2["subject"])
    o1, o2 = normalize(t1["object"]),   normalize(t2["object"])
    p1, p2 = get_lemma(t1.get("predicate","")), get_lemma(t2.get("predicate",""))
    subj_ok = s1 in s2 or s2 in s1 or any(w in s2.split() for w in s1.split() if len(w)>3)
    obj_ok  = o1 in o2 or o2 in o1 or any(w in o2.split() for w in o1.split() if len(w)>3)
    return subj_ok and (p1==p2) and obj_ok

def intersect(openie_triples, gpt_triples):
    seen, confirmed = set(), []
    for g in gpt_triples:
        sig = (normalize(g["subject"]), get_lemma(g.get("predicate","")), normalize(g["object"]))
        if sig in seen: continue
        if any(triples_match(g, o) for o in openie_triples):
            seen.add(sig)
            g["source"] = "openie+gpt"
            confirmed.append(g)
    return confirmed

In [5]:
# ============================================================
# Run + Build clause_predicate_dict
# ============================================================
gpt_index    = defaultdict(list)
for t in gpt_all:
    key = (t.get("contract","")[:50], t.get("clause_type",""), t.get("raw_text","")[:60])
    gpt_index[key].append(t)

result_triples = defaultdict(list)
seen_global    = set()

for contract in tqdm(data[:NUM_CONTRACTS], desc="Processing"):
    title         = contract["title"]
    clause_answers = {}
    for para in contract["paragraphs"]:
        for qa in para["qas"]:
            if qa["is_impossible"]: continue
            m = re.search(r'"([^"]+)"', qa["question"])
            if not m: continue
            ct = m.group(1)
            if ct in clause_answers: continue
            for answer in qa["answers"]:
                text = clean_text(answer["text"])
                if not is_valid(text) or len(text.split()) > 120: continue
                clause_answers[ct] = text
                break

    for ct, text in clause_answers.items():
        key       = (title[:50], ct, text[:60])
        confirmed = intersect(extract_openie(text, ct, title), gpt_index.get(key, []))
        for t in confirmed:
            sig = (title, normalize(t["subject"]), get_lemma(t.get("predicate","")), normalize(t["object"]))
            if sig in seen_global: continue
            seen_global.add(sig)
            t["contract"] = title
            t["clause_type"] = ct
            t["raw_text"] = text
            result_triples[ct].append(t)

# ── Build FLAT clause_predicate_dict ─────────────────────────
clause_predicate_flat = {}
for ct, triples in result_triples.items():
    predicates = sorted(set(t["predicate"] for t in triples))
    clause_predicate_flat[ct] = predicates

# ── Display ───────────────────────────────────────────────────
for ct in sorted(clause_predicate_flat.keys()):
    print(f"\n{ct}")
    print(f"{'─'*40}")
    for pred in clause_predicate_flat[ct]:
        print(f"  {pred}")

# ── Save ─────────────────────────────────────────────────────
with open("clause_predicate_dict.json", "w") as f:
    json.dump(clause_predicate_flat, f, indent=2)

print("\nSaved: clause_predicate_dict.json")

Processing: 100%|██████████| 50/50 [03:25<00:00,  4.11s/it]


Affiliate License-Licensee
────────────────────────────────────────
  may sublicense

Anti-Assignment
────────────────────────────────────────
  may assign
  may collaterally assign
  may make
  may not be assigned
  may transfer
  shall assign
  shall be required
  shall delegate
  shall deliver
  shall not assign
  shall provide
  shall require
  will require

Audit Rights
────────────────────────────────────────
  may appoint
  may audit
  may exercise
  shall bear
  shall furnish
  shall maintain
  shall reimburse
  shall use
  will be conducted
  will provide

Cap On Liability
────────────────────────────────────────
  shall exceed
  shall not exceed
  will be limited to

Change Of Control
────────────────────────────────────────
  may assign
  require
  shall have expressly consented
  shall notify
  will provide

Competitive Restriction Exception
────────────────────────────────────────
  will not restrict

Covenant Not To Sue
────────────────────────────────────────
  may cont